In [5]:
!pip -q install opencv-python numpy matplotlib torch torchvision pillow

In [6]:
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

LEFT_PATH = Path('/content/viprectification_deskLeft.png')
RIGHT_PATH = Path('/content/viprectification_deskRight.png')

if not LEFT_PATH.exists():
    from google.colab import drive
    drive.mount('/content/drive')
    roots = [p for p in Path('/content/drive/MyDrive').iterdir() if p.is_dir()]
    sid = Path('/content/drive/.shortcut-targets-by-id/1fUo6TQmvkCPWg9MVaEQTyx3tOZ0yTEkG')
    if sid.exists():
        roots.append(sid)
    l = next((p for r in roots for p in r.rglob('viprectification_deskLeft.png')), None)
    r = next((p for r in roots for p in r.rglob('viprectification_deskRight.png')), None)
    shutil.copy(l, LEFT_PATH)
    shutil.copy(r, RIGHT_PATH)

In [7]:
from PIL import Image
from torchvision.models import resnet50, ResNet50_Weights


def detect_corners(gray, max_corners=500):
    pts = cv2.goodFeaturesToTrack(gray, maxCorners=max_corners, qualityLevel=0.01, minDistance=8)
    return pts.reshape(-1, 2).astype(np.float64)


def extract_patch_features(image_bgr, keypoints, patch_size=8, model=None, transform=None, device='cpu'):
    h, w = image_bgr.shape[:2]
    half = patch_size // 2
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    feats, valid = [], []
    for x, y in keypoints:
        xi, yi = int(round(x)), int(round(y))
        if xi - half < 0 or yi - half < 0 or xi + half > w or yi + half > h:
            continue
        patch = rgb[yi - half:yi + half, xi - half:xi + half]
        img = Image.fromarray(patch).resize((224, 224))
        with torch.no_grad():
            feat = model(transform(img).unsqueeze(0).to(device)).cpu().numpy().reshape(-1)
        feats.append(feat)
        valid.append([x, y])
    return np.asarray(feats), np.asarray(valid, dtype=np.float64)


def match_features(feat_left, feat_right, ratio_thresh=0.75):
    matches = []
    for i, f1 in enumerate(feat_left):
        dists = np.linalg.norm(feat_right - f1, axis=1)
        order = np.argsort(dists)
        if len(order) < 2:
            continue
        if dists[order[0]] < ratio_thresh * dists[order[1]]:
            matches.append([i, order[0]])
    return np.asarray(matches, dtype=np.int64)


def normalize_points(pts):
    c = pts.mean(axis=0)
    d = np.sqrt(((pts - c) ** 2).sum(axis=1)).mean()
    s = np.sqrt(2) / (d + 1e-8)
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]], dtype=np.float64)
    pts_h = np.hstack([pts, np.ones((len(pts), 1))])
    return T, (T @ pts_h.T).T[:, :2]


def eight_point_fundamental(pts1, pts2):
    T1, n1 = normalize_points(pts1)
    T2, n2 = normalize_points(pts2)
    A = np.column_stack([
        n2[:, 0] * n1[:, 0], n2[:, 0] * n1[:, 1], n2[:, 0],
        n2[:, 1] * n1[:, 0], n2[:, 1] * n1[:, 1], n2[:, 1],
        n1[:, 0], n1[:, 1], np.ones(len(n1)),
    ])
    _, _, Vt = np.linalg.svd(A)
    F = Vt[-1].reshape(3, 3)
    U, S, Vt = np.linalg.svd(F)
    S[-1] = 0
    F = U @ np.diag(S) @ Vt
    F = T2.T @ F @ T1
    return F / (np.linalg.norm(F) + 1e-12)


def sampson_distance(F, pts1, pts2):
    p1 = np.hstack([pts1, np.ones((len(pts1), 1))])
    p2 = np.hstack([pts2, np.ones((len(pts2), 1))])
    Fp1 = (F @ p1.T).T
    FTp2 = (F.T @ p2.T).T
    num = np.sum(p2 * Fp1, axis=1) ** 2
    den = Fp1[:, 0] ** 2 + Fp1[:, 1] ** 2 + FTp2[:, 0] ** 2 + FTp2[:, 1] ** 2
    return num / (den + 1e-12)


def ransac_fundamental(pts1, pts2, n_iter=2000, sample_size=8):
    n = len(pts1)
    rng = np.random.default_rng(0)
    best_F, best_err = None, np.inf
    for _ in range(n_iter):
        idx = rng.choice(n, sample_size, replace=False)
        F = eight_point_fundamental(pts1[idx], pts2[idx])
        rest = np.ones(n, dtype=bool)
        rest[idx] = False
        err = sampson_distance(F, pts1[rest], pts2[rest]).mean()
        if err < best_err:
            best_F, best_err = F, err
    return best_F, best_err

In [8]:
img_l = cv2.imread(str(LEFT_PATH))
img_r = cv2.imread(str(RIGHT_PATH))
gray_l = cv2.cvtColor(img_l, cv2.COLOR_BGR2GRAY)
gray_r = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)

kp_l = detect_corners(gray_l)
kp_r = detect_corners(gray_r)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights).to(device).eval()
model.fc = torch.nn.Identity()
transform = weights.transforms()

feat_l, kp_l = extract_patch_features(img_l, kp_l, model=model, transform=transform, device=device)
feat_r, kp_r = extract_patch_features(img_r, kp_r, model=model, transform=transform, device=device)

matches = match_features(feat_l, feat_r)
pts_l = kp_l[matches[:, 0]]
pts_r = kp_r[matches[:, 1]]
print(f'Matched points: {len(pts_l)}')

F_best, mean_err = ransac_fundamental(pts_l, pts_r, n_iter=2000)
print('Best fundamental matrix F:')
print(F_best)
print(f'Mean Sampson error: {mean_err:.6f}')

Matched points: 48
Best fundamental matrix F:
[[-9.08511326e-06  1.00310634e-03 -1.66663073e-01]
 [-9.63109795e-04 -1.06204141e-05  2.02329772e-01]
 [ 1.76859614e-01 -2.24650099e-01  9.21703401e-01]]
Mean Sampson error: 88.948381
